# MathPhD++ : T4-Scale Mathematical Reasoning Prototype

**A complete 5-stage training pipeline for mathematical superintelligence on a single T4 GPU.**

| Stage | Description | Time on T4 |
|-------|-------------|------------|
| 1. CPT | Continued Pre-Training with multi-objective loss | ~8h |
| 2. SFT | Supervised Fine-Tuning on math instructions | ~6h |
| 3. PRM | Process Reward Model training | ~3h |
| 4. GRPO | Group Relative Policy Optimization with SymPy rewards | ~38h |
| 5. Eval | Benchmark evaluation + inference demos | ~3h |

**Base Model:** Qwen2.5-0.5B (494M params, full fine-tuning, ~7.4GB VRAM)

---

## 0. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets peft bitsandbytes trl accelerate sympy wandb sentencepiece einops tqdm

# Clone the MathPhD++ codebase (upload to Drive or clone from repo)
import os
import sys

# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory on Drive
GDRIVE_ROOT = '/content/drive/MyDrive/mathphd_checkpoints'
os.makedirs(GDRIVE_ROOT, exist_ok=True)

# If the code is uploaded to Drive:
CODE_PATH = '/content/drive/MyDrive/mathphd_plus_plus'
if os.path.exists(CODE_PATH):
    sys.path.insert(0, os.path.dirname(CODE_PATH))
else:
    # Alternatively, upload the mathphd_plus_plus folder to /content/
    print('Please upload the mathphd_plus_plus folder to /content/ or Google Drive')
    sys.path.insert(0, '/content')

# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Configuration
from mathphd_plus_plus.configs.base_config import MathPhDConfig

config = MathPhDConfig()
config.gdrive_checkpoint_root = GDRIVE_ROOT
config.use_wandb = False  # Set True if you have a wandb API key

# Reduce dataset sizes for quick testing (set to None for full training)
QUICK_TEST = True  # Set False for real training

if QUICK_TEST:
    config.cpt.target_tokens = 5_000_000  # 5M tokens instead of 500M
    config.cpt.num_train_epochs = 1
    config.cpt.save_steps = 50
    config.sft.num_train_epochs = 1
    config.prm.num_train_epochs = 1
    config.grpo.num_grpo_epochs = 1
    config.grpo.problems_per_epoch = 100

print('Config loaded. QUICK_TEST =', QUICK_TEST)

## 1. Load Base Model & Tokenizer

In [ ]:
from mathphd_plus_plus.models.base_model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    model_name=config.model.model_name,
    special_tokens=config.model.special_tokens,
    gradient_checkpointing=config.model.gradient_checkpointing,
)

# Memory check
allocated = torch.cuda.memory_allocated() / 1e9
print(f'GPU memory allocated: {allocated:.2f} GB')

## 2. Download & Prepare Data

In [ ]:
from mathphd_plus_plus.data.download import download_sft_data, download_prm_data, download_eval_data

CACHE_DIR = '/content/data_cache'

# Download datasets (SFT + PRM + Eval first, CPT streaming later)
print('Downloading SFT datasets...')
sft_raw = download_sft_data(cache_dir=CACHE_DIR)

print('\nDownloading PRM data...')
prm_raw = download_prm_data(cache_dir=CACHE_DIR)

print('\nDownloading eval data...')
eval_raw = download_eval_data(cache_dir=CACHE_DIR)

print('\nAll downloads complete!')

In [ ]:
# Prepare SFT dataset
from mathphd_plus_plus.data.preprocess_sft import prepare_sft_dataset

sft_dataset = prepare_sft_dataset(
    sft_raw,
    tokenizer=tokenizer,
    config=config.sft,
    max_seq_length=config.sft.max_seq_length,
)
print(f'\nSFT dataset ready: {len(sft_dataset)} samples')
print(f'Sample: {sft_dataset[0]["text"][:200]}...')

In [ ]:
# Prepare GRPO dataset
from mathphd_plus_plus.data.preprocess_grpo import prepare_grpo_dataset

grpo_dataset = prepare_grpo_dataset(
    sft_raw,
    max_problems=config.grpo.problems_per_epoch,
)
print(f'\nGRPO dataset ready: {len(grpo_dataset)} verifiable problems')

In [ ]:
# Prepare PRM dataset
from mathphd_plus_plus.data.preprocess_prm import prepare_prm_dataset

prm_max = 10000 if QUICK_TEST else 100000
prm_dataset = prepare_prm_dataset(prm_raw, max_samples=prm_max)
print(f'\nPRM dataset ready: {len(prm_dataset)} step examples')

## 3. Stage 1: Continued Pre-Training (CPT)

Multi-objective loss: `L_total = L_NTP + 0.1 * L_structure`

**Skip this stage if short on time** — go directly to SFT for faster results.

In [ ]:
SKIP_CPT = True  # Set False to run continued pre-training

if not SKIP_CPT:
    from mathphd_plus_plus.data.download import download_cpt_data
    from mathphd_plus_plus.data.preprocess_cpt import prepare_cpt_dataset
    from mathphd_plus_plus.training.cpt_trainer import run_cpt

    # Download streaming CPT data
    cpt_raw = download_cpt_data(cache_dir=CACHE_DIR)

    # Prepare
    cpt_dataset = prepare_cpt_dataset(
        cpt_raw,
        tokenizer=tokenizer,
        max_seq_length=config.cpt.max_seq_length,
        target_tokens=config.cpt.target_tokens,
    )

    # Train
    cpt_path = run_cpt(
        model=model,
        tokenizer=tokenizer,
        train_dataset=cpt_dataset,
        config=config.cpt,
        output_dir=os.path.join(GDRIVE_ROOT, 'cpt'),
    )
    print(f'CPT complete: {cpt_path}')
else:
    print('Skipping CPT stage (using base model directly for SFT)')

## 4. Stage 2: Supervised Fine-Tuning (SFT)

ChatML format with `<thinking>`, `<answer>` structure. Response-only loss masking.

In [ ]:
from mathphd_plus_plus.training.sft_trainer import run_sft

sft_path = run_sft(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    config=config.sft,
    output_dir=os.path.join(GDRIVE_ROOT, 'sft'),
)
print(f'\nSFT complete: {sft_path}')

In [ ]:
# Quick SFT quality check
test_problem = 'What is the sum of the first 100 positive integers?'

from mathphd_plus_plus.evaluation.evaluate import generate_answer
response = generate_answer(model, tokenizer, test_problem)
print(f'Problem: {test_problem}')
print(f'Response:\n{response}')

## 5. Stage 3: Process Reward Model (PRM)

Trains a step-level reward model for scoring reasoning quality.

In [ ]:
from mathphd_plus_plus.training.prm_trainer import run_prm_training

prm_path = run_prm_training(
    tokenizer=tokenizer,
    train_dataset=prm_dataset,
    config=config.prm,
    output_dir=os.path.join(GDRIVE_ROOT, 'prm'),
)
print(f'\nPRM complete: {prm_path}')

In [ ]:
# Load PRM for GRPO rewards
from mathphd_plus_plus.models.reward_model import ProcessRewardModel
from mathphd_plus_plus.rewards.process_reward import ProcessRewardScorer

prm_model = ProcessRewardModel(model_name=config.prm.model_name)
prm_state = torch.load(os.path.join(prm_path, 'prm_model.pt'), map_location='cpu')
prm_model.load_state_dict(prm_state)
prm_model = prm_model.to('cuda')

process_scorer = ProcessRewardScorer(prm_model, tokenizer, device='cuda')
print('PRM loaded for scoring')

## 6. Stage 4: GRPO Training

**The core innovation**: Group Relative Policy Optimization with:
- SymPy-verified correctness rewards
- PRM process rewards
- Format compliance rewards

This is the longest stage (~38h for full training, ~2h for quick test).

In [ ]:
from mathphd_plus_plus.rewards.composite_reward import CompositeReward
from mathphd_plus_plus.training.grpo_trainer import run_grpo

# Create composite reward function
reward_fn = CompositeReward(
    process_scorer=process_scorer,
    correctness_weight=config.grpo.reward_correctness_weight,
    process_weight=config.grpo.reward_process_weight,
    format_weight=config.grpo.reward_format_weight,
)

# Run GRPO
grpo_path = run_grpo(
    model=model,
    tokenizer=tokenizer,
    train_dataset=grpo_dataset,
    reward_fn=reward_fn,
    config=config.grpo,
    output_dir=os.path.join(GDRIVE_ROOT, 'grpo'),
)
print(f'\nGRPO complete: {grpo_path}')

## 7. Evaluation

Run benchmarks: GSM8K and MATH with difficulty/subject breakdown.

In [ ]:
from mathphd_plus_plus.evaluation.evaluate import run_all_evaluations

eval_max = 200 if QUICK_TEST else None

eval_results = run_all_evaluations(
    model=model,
    tokenizer=tokenizer,
    benchmarks=['gsm8k', 'math'],
    max_samples=eval_max,
    output_dir=os.path.join(GDRIVE_ROOT, 'eval_results'),
)

## 8. Inference Strategy Demos

Compare: Direct | Self-Consistency | Tree-of-Thoughts | MCTS | Multi-Agent Debate

In [ ]:
from mathphd_plus_plus.inference.pipeline import InferencePipeline

pipeline = InferencePipeline(
    model=model,
    tokenizer=tokenizer,
    process_scorer=process_scorer,
    config=config.inference,
)

# Test problems of varying difficulty
test_problems = [
    {
        'problem': 'What is 15% of 240?',
        'answer': '36',
        'difficulty': 1,
    },
    {
        'problem': 'Find the remainder when 2^100 is divided by 7.',
        'answer': '2',
        'difficulty': 3,
    },
    {
        'problem': 'Let $a, b, c$ be positive reals with $a+b+c=1$. Prove that $a^2+b^2+c^2 \\geq \\frac{1}{3}$.',
        'answer': '1/3',
        'difficulty': 4,
    },
]

In [ ]:
# Compare strategies
for tp in test_problems:
    print(f"\n{'='*60}")
    print(f"Problem (difficulty {tp['difficulty']}): {tp['problem'][:100]}")
    print(f"Ground truth: {tp['answer']}")
    print(f"{'='*60}")

    for strategy in ['direct', 'sc', 'tot']:
        try:
            result = pipeline.solve(
                tp['problem'],
                strategy=strategy,
                ground_truth=tp['answer'],
            )
            print(f"\n  [{strategy.upper()}] Answer: {result['answer']}")
            if 'confidence' in result:
                print(f"    Confidence: {result['confidence']:.2f}")
        except Exception as e:
            print(f"\n  [{strategy.upper()}] Error: {e}")

## 9. Conjecture Generation (Bonus)

Adversarial generator-critic loop to discover novel mathematical conjectures.

In [ ]:
from mathphd_plus_plus.inference.conjecture_generator import run_conjecture_generation

conjectures = run_conjecture_generation(
    model=model,
    tokenizer=tokenizer,
    num_conjectures=5,  # Small for demo
    domains=['number theory', 'combinatorics', 'linear algebra'],
)

print(f"\nInteresting (unresolved) conjectures:")
for c in conjectures.get('interesting', []):
    print(f"  - {c['conjecture'][:200]}")

print(f"\nResolved conjectures:")
for c in conjectures.get('resolved', [])[:3]:
    print(f"  - [{c['resolution']}] {c['conjecture'][:150]}")

## 10. Save Final Model

Save the fully trained model to Google Drive.

In [ ]:
# Save final model
final_model_path = os.path.join(GDRIVE_ROOT, 'mathphd_final')
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f'Final model saved to: {final_model_path}')

# Optional: push to HuggingFace Hub
# from huggingface_hub import login
# login()
# model.push_to_hub('your-username/mathphd-plus-plus-0.5b')
# tokenizer.push_to_hub('your-username/mathphd-plus-plus-0.5b')